# Textbook Tutor on Google Colab

Runs your full app (including Bengali EasyOCR) on Google's free cloud. Login with your Gmail — no credit card.

## How to use (read this first)
1. Tap **Runtime -> Run all**
2. A Google popup opens asking for Drive access -> pick your Gmail -> **Allow** (you have 2 minutes)
3. Wait. The last cell prints the shareable student link
4. Share the link. Manage at `LINK/admin` (default password `admin123`)

## Important limits (free Colab)
- A session lasts at most **~12 hours**, then ends. To continue: reopen the notebook, **Run all** again — your data is restored from Google Drive.
- **Keep the browser tab open** and the phone screen awake during class. If the phone sleeps or the browser is killed, students lose access.
- Use **Wi-Fi**, not mobile data (the first run downloads ~300MB of OCR models).
- Google can cut off free sessions that run websites 24/7. Treat this as a daily "start your server" routine.

If the Drive popup is blocked, the notebook continues without Drive (see the last two cells for manual backup/restore).


In [ ]:
import os, threading, time
from google.colab import drive

DRIVE_DATA = None
_done = threading.Event()

def _mount():
    try:
        drive.mount("/content/drive", force_remount=False)
    finally:
        _done.set()

threading.Thread(target=_mount, daemon=True).start()

print("\n=== ACTION NEEDED ===")
print("A 'Permit access to your Drive' window opened in a new tab.")
print("Pick your Gmail account, then tap ALLOW. You have 2 minutes.")
print("========================\n")

for waited in range(0, 120, 5):
    if os.path.isdir("/content/drive/MyDrive"):
        DRIVE_DATA = "/content/drive/MyDrive/textbook-tutor-data"
        os.makedirs(DRIVE_DATA, exist_ok=True)
        print("Google Drive connected. Data folder:", DRIVE_DATA)
        break
    if _done.is_set() and not os.path.isdir("/content/drive/MyDrive"):
        print("Drive mount finished but MyDrive was not found. Continuing without Drive.")
        break
    time.sleep(5)
    if waited in (30, 60, 90):
        print("Still waiting for the Google popup... (check the new tab)")
else:
    print("Drive did not connect in 2 minutes (popup blocked?).")
    print("Continuing WITHOUT auto-save. Use the manual backup cells at the end.")

In [ ]:
import os, subprocess

REPO = "/content/textbook-tutor"
LOCAL_DATA = "/content/tt-data"
os.makedirs(REPO, exist_ok=True)
os.makedirs(LOCAL_DATA, exist_ok=True)

if not os.listdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/nobnoob001-ops/textbook-tutor.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "pull"], capture_output=True)

os.environ["DATA_DIR"] = LOCAL_DATA
os.environ["EASYOCR_MODULE_PATH"] = "/content/.EasyOCR"
print("Repo ready. Runtime data folder:", LOCAL_DATA)

In [ ]:
import subprocess, sys

print("Installing system tools + Python packages (2-5 min, one-time)...", flush=True)
subprocess.run("apt-get update -qq && apt-get install -qq -y tesseract-ocr tesseract-ocr-ben poppler-utils > /dev/null 2>&1", shell=True)
r = subprocess.run("pip -q install easyocr==1.7.2 fastapi uvicorn httpx pypdf pdf2image pytesseract numpy Pillow opencv-python-headless 2>&1 | tail -n 2", shell=True, capture_output=True, text=True)
print("Dependencies installed.", r.stdout.strip())

In [ ]:
import glob, os, shutil

def _cp(src, dst):
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(src, dst)

if DRIVE_DATA:
    src_db = os.path.join(DRIVE_DATA, "tutor.db")
    if os.path.exists(src_db):
        shutil.copy(src_db, os.path.join(LOCAL_DATA, "tutor.db"))
        print("Restored database from Drive.")
    for p in glob.glob(os.path.join(DRIVE_DATA, ".EasyOCR", "**", "*"), recursive=True):
        if os.path.isfile(p):
            rel = os.path.relpath(p, os.path.join(DRIVE_DATA, ".EasyOCR"))
            _cp(p, os.path.join("/content/.EasyOCR", rel))
    print("Restored data from Drive.")
else:
    print("Drive not connected - nothing to restore.")

In [ ]:
import os, sqlite3, threading, time

def backup_db():
    if not DRIVE_DATA:
        return
    src = os.path.join(LOCAL_DATA, "tutor.db")
    if not os.path.exists(src):
        return
    tmp = os.path.join(DRIVE_DATA, "tutor.db.tmp")
    dst = os.path.join(DRIVE_DATA, "tutor.db")
    try:
        a = sqlite3.connect(src)
        b = sqlite3.connect(tmp)
        a.backup(b)
        b.close()
        a.close()
        os.replace(tmp, dst)
    except Exception as e:
        print("backup error:", e)

def _loop(interval):
    while True:
        time.sleep(interval)
        backup_db()

threading.Thread(target=_loop, args=(60,), daemon=True).start()
if DRIVE_DATA:
    print("Autosave ON: database -> Google Drive every 60s.")
else:
    print("Autosave OFF (Drive not connected). Use manual backup at the end.")

In [ ]:
import os, socket, subprocess, time, httpx

subprocess.run("pkill -9 -f run.py; pkill -9 -f uvicorn; pkill -9 -f cloudflared", shell=True, capture_output=True)

def _port_free(port):
    s = socket.socket()
    try:
        s.bind(("0.0.0.0", port))
        return True
    except OSError:
        return False
    finally:
        s.close()

for _ in range(20):
    if _port_free(8080):
        break
    time.sleep(1)
else:
    print("WARNING: port 8080 still busy after 20s")

os.chdir(REPO)
env = dict(os.environ, HOST="0.0.0.0", PORT="8080")
with open("/content/tt-server.log", "w") as log:
    subprocess.Popen(
        ["python", "run.py"],
        stdout=log, stderr=subprocess.STDOUT, env=env,
    )

up = False
for _ in range(60):
    time.sleep(2)
    try:
        r = httpx.get("http://localhost:8080/", timeout=3)
        if r.status_code == 200:
            up = True
            break
    except Exception:
        pass
print("Server up:", up)
if not up:
    print(open("/content/tt-server.log").read()[-2000:])

In [ ]:
import glob, os, shutil, subprocess

models_dir = "/content/.EasyOCR"
if not os.path.isdir(os.path.join(models_dir, "model")):
    print("Downloading OCR models (one-time, ~300MB). Keep the tab open...", flush=True)
    subprocess.run(["python", "-c", "import easyocr; easyocr.Reader(['bn','en'], gpu=False)"], check=True)
    if DRIVE_DATA:
        for p in glob.glob(os.path.join(models_dir, "**", "*"), recursive=True):
            if os.path.isfile(p):
                rel = os.path.relpath(p, models_dir)
                dst = os.path.join(DRIVE_DATA, ".EasyOCR", rel)
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                shutil.copy2(p, dst)
    print("OCR models ready" + (" and cached to Drive." if DRIVE_DATA else "."))
else:
    print("OCR models already present.")

In [ ]:
import os, re, subprocess, time

def ensure_cloudflared():
    if not os.path.exists("/usr/local/bin/cloudflared"):
        subprocess.run(["wget", "-q",
            "https://github.com/cloudflare/cloudflared/releases/download/2026.7.3/cloudflared-linux-amd64",
            "-O", "/usr/local/bin/cloudflared"], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])

ensure_cloudflared()
with open("/content/tt-tunnel.log", "w") as log:
    subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:8080", "--no-autoupdate"],
        stdout=log, stderr=subprocess.STDOUT)

URL = None
for _ in range(60):
    time.sleep(2)
    try:
        text = open("/content/tt-tunnel.log").read()
    except Exception:
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
    if m:
        URL = m.group(0)
        break
print("STUDENT LINK:", URL)
print("ADMIN LINK:", URL + "/admin")
if not URL:
    print(open("/content/tt-tunnel.log").read()[-2000:])

In [ ]:
import time, httpx

print("")
print("==================== SHARE THIS LINK WITH STUDENTS ====================")
print("  " + str(URL))
print("  Admin (password): " + str(URL) + "/admin")
print("=========================================================================")
print("This cell keeps the session alive (prevents idle disconnect).")
print("Keep this tab open. Close the tab to stop the server.")
print("Database saves to Drive every 60s.")
print("")
try:
    httpx.get("http://localhost:8080/", timeout=5)
except Exception:
    pass
while True:
    time.sleep(60)
    backup_db()
    try:
        httpx.get("http://localhost:8080/", timeout=5)
    except Exception:
        pass

## Done!
Keep the last cell running. If students lose access, reopen this notebook and tap **Runtime -> Run all** again.

### Manual backup (only if Drive failed to connect)
Run the **Download backup** cell to save your data, and the **Restore backup** cell next time.


In [ ]:
from google.colab import files
import os
# Downloads your database so you can save it somewhere safe
files.download(os.path.join(LOCAL_DATA, "tutor.db"))
print("tutor.db downloaded. Keep it. Next session, run the 'Restore' cell.")

In [ ]:
from google.colab import files
import os
# Upload the tutor.db file you saved earlier
up = files.upload()
for name in up:
    with open(os.path.join(LOCAL_DATA, "tutor.db"), "wb") as f:
        f.write(up[name])
    print("Restored", name)